In [1]:
!pip install requests pandas numpy scikit-learn

In [2]:
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# ==========================================
# 1. NASA POWER API THEKE DATA FETCH KORA
# ==========================================


def fetch_nasa_power_data(
    lat=23.8103, lon=90.4125, start="20230101", end="20231231"
):
  """NASA POWER Agroclimatology API theke weather & soil moisture data ane.

  lat, lon: Location coordinates (Default: Dhaka)
  """
  url = "https://power.larc.nasa.gov/api/temporal/daily/point"

  # Parameters:
  # T2M: Temperature at 2m (C)
  # PRECTOTCORR: Precipitation (mm/day)
  # GWETTOP: Top-layer Soil Wetness (0 to 1, where 1 is saturated)
  # RH2M: Relative Humidity at 2m (%)
  params = {
      "parameters": "T2M,PRECTOTCORR,GWETTOP,RH2M",
      "community": "AG",
      "longitude": lon,
      "latitude": lat,
      "start": start,
      "end": end,
      "format": "JSON",
  }

  print(f"Fetching data from NASA POWER API for Lat: {lat}, Lon: {lon}...")
  response = requests.get(url, params=params)

  if response.status_code != 200:
    raise Exception(f"API Error: {response.status_code}, {response.text}")

  data = response.json()
  records = data["properties"]["parameter"]

  # DataFrame toiri kora
  df = pd.DataFrame({
      "Temperature": records["T2M"],
      "Precipitation": records["PRECTOTCORR"],
      "SoilMoisture": records["GWETTOP"],
      "Humidity": records["RH2M"],
  })

  df.index = pd.to_datetime(df.index)
  df.sort_index(inplace=True)
  return df


# ==========================================
# 2. FEATURE ENGINEERING & PREPROCESSING
# ==========================================


def prepare_features(df):
  # Missing values ba fill values (-999) thakle clean kora
  df.replace(-999, np.nan, inplace=True)
  df.ffill(inplace=True)

  # Rolling window features (3-day & 7-day average)
  df["Rain_3d_sum"] = df["Precipitation"].rolling(window=3).sum()
  df["Temp_3d_avg"] = df["Temperature"].rolling(window=3).mean()
  df["Humidity_3d_avg"] = df["Humidity"].rolling(window=3).mean()

  # Target Variable: Next day Soil Moisture Risk Level
  # Rule: GWETTOP < 0.35 mane High Risk / Drought Stress (1), onnothay Safe (0)
  df["Next_Day_SoilMoisture"] = df["SoilMoisture"].shift(-1)
  df["Drought_Risk_Target"] = (df["Next_Day_SoilMoisture"] < 0.35).astype(int)

  # Drop null values generated from shift and rolling
  df.dropna(inplace=True)
  return df


# ==========================================
# 3. BASELINE ML MODEL TRAINING
# ==========================================


def train_baseline_model(df):
  feature_cols = [
      "Temperature",
      "Precipitation",
      "SoilMoisture",
      "Humidity",
      "Rain_3d_sum",
      "Temp_3d_avg",
      "Humidity_3d_avg",
  ]

  X = df[feature_cols]
  y = df["Drought_Risk_Target"]

  # Time-series aware split (shuffle=False)
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, shuffle=False
  )

  model = RandomForestClassifier(n_estimators=100, random_state=42)
  model.fit(X_train, y_train)

  # Evaluation
  y_pred = model.predict(X_test)
  print("\n--- Model Performance on Test Data ---")
  print(classification_report(y_test, y_pred))

  return model, feature_cols


# ==========================================
# 4. REAL-TIME ADVISORY PREDICTION
# ==========================================


def generate_advisory(latest_data_row, model, feature_cols):
  prediction = model.predict([latest_data_row[feature_cols]])[0]
  prob = model.predict_proba([latest_data_row[feature_cols]])[0][1]

  print(f"\nSoil Moisture Risk Probability: {prob * 100:.1f}%")
  if prediction == 1 or prob >= 0.5:
    print(
        "Advisory Status: HIGH RISK (Krishokder jonne shobar aage shech"
        " [Irrigation] dewar paramorsho dewya hocche)."
    )
  else:
    print(
        "Advisory Status: NORMAL (Maatir aadrota shavabik ache, baroti shecher"
        " proyojon nei)."
    )


# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
  # 1 bochorer data load kora
  raw_df = fetch_nasa_power_data(
      lat=23.8103, lon=90.4125, start="20230101", end="20231231"
  )
  processed_df = prepare_features(raw_df)

  # Model train kora
  trained_model, features = train_baseline_model(processed_df)

  # Latest day data diye advisory test kora
  latest_record = processed_df.iloc[-1]
  generate_advisory(latest_record, trained_model, features)

Fetching data from NASA POWER API for Lat: 23.8103, Lon: 90.4125...

--- Model Performance on Test Data ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        73

    accuracy                           1.00        73
   macro avg       1.00      1.00      1.00        73
weighted avg       1.00      1.00      1.00        73


Soil Moisture Risk Probability: 0.0%
Advisory Status: NORMAL (Maatir aadrota shavabik ache, baroti shecher proyojon nei).


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [3]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 31.0 MB/s eta 0:00:00


In [6]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import requests
from sklearn.ensemble import RandomForestClassifier

# Page configuration
st.set_page_config(page_title="AgroShift - NASA Farm Advisor", layout="wide")

st.title("🌾 AgroShift: AI Climate & Soil Moisture Advisor")
st.markdown("Powered by NASA POWER Agroclimatology Data | Predicting Soil Moisture Risk & Irrigation Needs")

# Sidebar Configuration
st.sidebar.header("📍 Farm Coordinates")
lat = st.sidebar.number_input("Latitude", value=23.8103, format="%.4f")
lon = st.sidebar.number_input("Longitude", value=90.4125, format="%.4f")

@st.cache_data
def get_data(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M,PRECTOTCORR,GWETTOP,RH2M",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": "20230101",
        "end": "20231231",
        "format": "JSON"
    }
    res = requests.get(url, params=params).json()
    p = res['properties']['parameter']
    df = pd.DataFrame({
        'Temperature (°C)': p['T2M'],
        'Rainfall (mm)': p['PRECTOTCORR'],
        'SoilMoisture': p['GWETTOP'],
        'Humidity (%)': p['RH2M']
    })
    df.index = pd.to_datetime(df.index)
    df.replace(-999, np.nan, inplace=True)
    df.ffill(inplace=True)
    return df

with st.spinner("Fetching Satellite Data from NASA API..."):
    df = get_data(lat, lon)

# Current Metrics
col1, col2, col3, col4 = st.columns(4)
latest = df.iloc[-1]

col1.metric("Current Temperature", f"{latest['Temperature (°C)']:.1f} °C")
col2.metric("Precipitation", f"{latest['Rainfall (mm)']:.1f} mm")
col3.metric("Soil Moisture", f"{latest['SoilMoisture']*100:.1f}%")
col4.metric("Relative Humidity", f"{latest['Humidity (%)']:.1f}%")

st.divider()

# ML Processing & Prediction
df['Target'] = (df['SoilMoisture'].shift(-1) < 0.35).astype(int)
df_clean = df.dropna()

features = ['Temperature (°C)', 'Rainfall (mm)', 'SoilMoisture', 'Humidity (%)']
X = df_clean[features]
y = df_clean['Target']

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X, y)

prob = model.predict_proba([latest[features]])[0][1]

# Display Prediction & Advisory
st.subheader("🚨 AI Drought Stress & Soil Moisture Prediction")
col_a, col_b = st.columns([1, 2])

with col_a:
    st.metric("Predicted Drought Risk Score", f"{prob * 100:.1f}%")

with col_b:
    if prob >= 0.5:
        st.error("⚠️ **High Risk Alert:** Soil moisture is critically low. Immediate irrigation is recommended to prevent crop damage.")
    else:
        st.success("✅ **Optimal Condition:** Soil moisture is within safe levels. No additional irrigation required today.")

st.divider()

# Trend Visualizations
st.subheader("📊 Historical Trends: Soil Moisture vs Rainfall")
st.line_chart(df[['SoilMoisture', 'Rainfall (mm)']].tail(60))

Overwriting app.py


In [7]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501

136.87.4.236
⠙⠹⠸

⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-09-27 14:33:54.874 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.87.4.236:8501

  Stopping...
^C


In [8]:
!pip install -q pyngrok
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb
!streamlit run app.py & cloudflared tunnel --url http://localhost:8501

Selecting previously unselected package cloudflared.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...
2026-09-27T14:37:40Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-27T14:37:40Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-09-27 14:37:41.953 Uv